In [5]:
import requests
import pandas as pd
import ast

from APIKey import TMDB_API_KEY

BASE_URL = "https://api.themoviedb.org/3"

# Load genre lookup without hardcoding
df_genres = pd.read_csv("genres.csv") # Listed id, genre name
genre_lookup = dict(zip(df_genres["id"], df_genres["name"])) # Combine ID and word into a list


In [ ]:
def search_movie(term):
    url = f"{BASE_URL}/search/movie"
    params = {
        "api_key": TMDB_API_KEY,
        "query": term
    }
    response = requests.get(url, params=params)

    # Check to make sure results are actually being sent
    if response.status_code != 200:
        print("Error: ", response.status_code)
        print(response.text)
        return []


    data = response.json()

    # If something is returned that isn't expected
    if "results" not in data or data["results"] is None:
        print("TMDB returned no results.")
        print(data)
        return []
    
    results = data.get("results", [])

    movies = []
    for m in results:
        movies.append({
            "id": m["id"],
            "title": m["title"],
            "year": (m.get("release_date") or "")[:4],
            "overview": m.get("overview", ""),
            "genre_ids": m.get("genre_ids", [])
        })



    return movies

In [4]:
# Testing Movie lookup without hardcoding
term = input("Search for a movie: ")
movies = search_movie(term)
print(movies[:3])

[{'id': 679, 'title': 'Aliens', 'year': '1986', 'overview': "Ripley, the sole survivor of the Nostromo's deadly encounter with the monstrous Alien, returns to Earth after drifting through space in hypersleep for 57 years. Although her story is initially met with skepticism, she agrees to accompany a team of Colonial Marines back to LV-426.", 'genre_ids': [28, 53, 878]}, {'id': 49849, 'title': 'Cowboys & Aliens', 'year': '2011', 'overview': 'A stranger stumbles into the desert town of Absolution with no memory of his past and a futuristic shackle around his wrist. With the help of mysterious beauty Ella and the iron-fisted Colonel Dolarhyde, he finds himself leading an unlikely posse of cowboys, outlaws, and Apache warriors against a common enemy from beyond this world in an epic showdown for survival.', 'genre_ids': [28, 878, 53, 37]}, {'id': 15512, 'title': 'Monsters vs Aliens', 'year': '2009', 'overview': 'When Susan Murphy is unwittingly clobbered by a meteor full of outer space gun

In [5]:
def submit_review(movie_id, rating, genre_ids=None):
    # Validate User review
    if rating < 1 or rating > 5:
        print("You must pick between 1 and 5 stars.")
        return
    
    # Load existing reviews (if any)
    try:
        df = pd.read_csv("reviews.csv")
    except FileNotFoundError:
        df = pd.DataFrame(columns=["movie_id", "rating", "genre_ids"])

    # Add a new row for the new review
    new_row = {
        "movie_id": movie_id,
        "rating": rating,
        "genre_ids": str(genre_ids) if genre_ids is not None else "" # Add empty string if no genre id string to keep csv clean
    }
    # Add new review to the bottom of reviews.csv
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    # Save new review to reviews.csv
    df.to_csv("reviews.csv", index=False)
    print("Your review was saved!")

In [7]:
# Test submit_review without hardcoding
term = input("Search for a movie: ")
movies = search_movie(term)

# Show search results
for i, m in enumerate(movies):
    print(f"{i+1}. {m['title']} ({m['year']})")

# User selects one from list of results
choice = int(input("Select a movie from the list using the number: "))
selected = movies[choice]

# User enters rating
rating = int(input("Enter your star review (1-5): "))

# Submit review
submit_review(selected["id"], rating, selected["genre_ids"])


1. Aliens (1986)
2. Cowboys & Aliens (2011)
3. Monsters vs Aliens (2009)
4. Aliens vs Predator: Requiem (2007)
5. Aliens (1982)
6. Aliens (2016)
7. Aliens in the Attic (2009)
8. The Aliens (2018)
9. Aliens (1993)
10. Smoking Aliens (2018)
11. Illegal Aliens (2016)
12. Luis and the Aliens (2018)
13. Aliens Expanded (2024)
14. Resident Aliens (2012)
15. Illegal Aliens (2007)
16. Aliens, Clowns & Geeks (2021)
17. Evil Aliens (2006)
18. Kids vs. Aliens (2023)
19. Aliens (2007)
20. Mutant Aliens (2002)
Your review was saved!


In [8]:
def get_movie_details(movie_id):

    # Get movie details from TMDB
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {
        "api_key": TMDB_API_KEY
    }
    response = requests.get(url, params=params)

    # Check to make sure request is fufilled by TMDB API
    if response.status_code != 200:
        print("TMDB request error:", response.status_code)
        return {}
    
    # Return details in JSON format
    return response.json()

In [9]:
# Testing get_movie_details
term = input("Search for a movie: ")
movies = search_movie(term)

# Show results of search
for i, m in enumerate(movies):
    print(f"{i+1}. {m['title']} ({m['year']})")

# Have user pick one to view details
selected = movies[int(input("Select a movie by the number on the list: ")) - 1]
details = get_movie_details(selected["id"])

print(details)

1. Aliens (1986)
2. Cowboys & Aliens (2011)
3. Monsters vs Aliens (2009)
4. Aliens vs Predator: Requiem (2007)
5. Aliens (1982)
6. Aliens (2016)
7. Aliens in the Attic (2009)
8. The Aliens (2018)
9. Aliens (1993)
10. Smoking Aliens (2018)
11. Illegal Aliens (2016)
12. Luis and the Aliens (2018)
13. Aliens Expanded (2024)
14. Resident Aliens (2012)
15. Illegal Aliens (2007)
16. Aliens, Clowns & Geeks (2021)
17. Evil Aliens (2006)
18. Kids vs. Aliens (2023)
19. Aliens (2007)
20. Mutant Aliens (2002)
{'adult': False, 'backdrop_path': None, 'belongs_to_collection': None, 'budget': 0, 'genres': [], 'homepage': '', 'id': 501123, 'imdb_id': 'tt3154258', 'origin_country': ['JP'], 'original_language': 'ja', 'original_title': 'エイリアンズ', 'overview': 'Rica, a woman who lost her memory is found by the lakeside. Staying with her discoverer Ataru, she wanders about the town seeking for clues to her past. While Rica gets back some framented memories, such as the experience as a hooker and her own pregn

In [2]:
def movies_with_genre(df_movies, genre_id):
    # Filter results by user selected genre
    return df_movies[df_movies["genre_ids"].apply(lambda g: genre_id in g)]

In [7]:
# Test movies_with_genre() filtering
term = input("Search for a movie: ")
movies = search_movie(term)
df_movies = pd.DataFrame(movies)
display(df_movies)

# Prompt user for genre for filtering
genre_name = input("Enter a genre to filter by: ")

# Load genre lookup
df_genres = pd.read_csv("genres.csv")
df_genres.columns = df_genres.columns.str.strip()

# Convert names back to ID
genre_id = df_genres[df_genres["name"] == genre_name]["id"].iloc[0]

# Filter movies by selected genre
filtered = movies_with_genre(df_movies, genre_id)
display(filtered)

,id,title,year,overview,genre_ids
0,679,Aliens,1986,"Ripley, the sole survivor of the Nostromo's de...","[28, 53, 878]"
1,49849,Cowboys & Aliens,2011,A stranger stumbles into the desert town of Ab...,"[28, 878, 53, 37]"
2,15512,Monsters vs Aliens,2009,When Susan Murphy is unwittingly clobbered by ...,"[16, 10751, 12, 878]"
3,440,Aliens vs Predator: Requiem,2007,After a horrifying PredAlien crash-lands near ...,"[28, 878, 53, 27]"
4,760646,Aliens,1982,In June 1940 Italy entered the war alongside G...,"[10752, 18, 10770]"
5,643818,Aliens,2016,A young woman arrives in a city for the first ...,[18]
6,20856,Aliens in the Attic,2009,A group of kids must protect their vacation ho...,"[12, 35, 10751, 14, 878]"
7,523910,The Aliens,2018,A UFO believer must choose between the aliens ...,"[18, 878]"
8,336311,Aliens,1993,Two young boys discuss their favorite movies a...,[]
9,1159015,Smoking Aliens,2018,"Then one day, her daily life suddenly changes....","[878, 27]"


,id,title,year,overview,genre_ids
0,679,Aliens,1986,"Ripley, the sole survivor of the Nostromo's de...","[28, 53, 878]"
1,49849,Cowboys & Aliens,2011,A stranger stumbles into the desert town of Ab...,"[28, 878, 53, 37]"
3,440,Aliens vs Predator: Requiem,2007,After a horrifying PredAlien crash-lands near ...,"[28, 878, 53, 27]"


In [16]:
def generate_movieboard(top_n=10):
    # Read reviews.csv and check to see if there is reviews
    try:
        df = pd.read_csv("reviews.csv")
    except FileNotFoundError:
        print("No reviews yet.")
        return
    
    # No reviews present?
    if df.empty:
        print("No reviews yet.")
        return
    
    # Convert genre_ids back to lists
    df["genre_ids"] = df["genre_ids"].apply(lambda x: ast.literal_eval(x) if x else []) # Add space if no genre id is present
    
    # group movies by ID
    grouped_ID = df.groupby("movie_id")["rating"].agg(["mean", "count"]).reset_index()
    grouped_ID = grouped_ID.sort_values(by=["mean", "count"], ascending=[False, False])

    # Show top movies based on set value (n)
    print("Top movies:")
    for i, row in grouped_ID.head(top_n).iterrows():
        movie_id = int(row["movie_id"])
        avg = row["mean"]
        count = int(row["count"])

        # Get movie title from TMDB 
        details = get_movie_details(movie_id)
        title = details.get("title", "Unknown title")

        # Convert TMDB Genre IDs to words
        tmdb_genres = details.get("genres", [])
        genre_names = [genre_lookup[g["id"]] for g in tmdb_genres if g["id"] in genre_lookup]

        # Print movieboard entry
        print(f"{title} ({', '.join(genre_names)}) -- Avg: {avg:.2f} from {count} reviews")
    

In [17]:
# Test generate_movieboard
generate_movieboard()

Top movies:
Idiocracy (Comedy, Science Fiction, Adventure) -- Avg: 5.00 from 1 reviews
The Toxic Avenger: The Musical (Comedy, Horror, Science Fiction, Music) -- Avg: 5.00 from 1 reviews
Star Wars Dreams (TV Movie, Documentary) -- Avg: 5.00 from 1 reviews
Cowboys & Aliens (Action, Science Fiction, Thriller, Western) -- Avg: 4.00 from 1 reviews
Interstellar (Adventure, Drama, Science Fiction) -- Avg: 3.00 from 1 reviews


In [18]:
# Test csv loading
df = pd.read_csv("reviews.csv")
print(df)

   movie_id  rating             genre_ids
0      7512     NaN         [35, 878, 12]
1      7512     NaN         [35, 878, 12]
2      7512     5.0         [35, 878, 12]
3    157336     3.0         [12, 18, 878]
4    657296     5.0           [10770, 99]
5    517289     5.0  [35, 27, 878, 10402]
6     49849     4.0     [28, 878, 53, 37]
